# Features and Model Panels

Construct S0/S1/S2 features, SPY-relative targets and sample eligibility from cleaned market data. Export the feature tables, shared model metadata and 27 sector-specific predictor files.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
from itertools import combinations
import hashlib
import json
import platform
import numpy as np
import pandas as pd
from IPython.display import display

ROOT = next((p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
             if (p / "requirements.txt").is_file()
             and (p / "notebooks").is_dir() and (p / "scripts").is_dir()), None)
if ROOT is None:
    raise FileNotFoundError("Open this notebook inside the project directory or a subdirectory.")
if not (ROOT / "notebooks/cleaning_checks/cleaning_manifest.json").is_file():
    raise FileNotFoundError("Run notebook 01 to completion before notebook 02.")
INTERIM = ROOT / "data/interim"
FEATURE_REPORTS = ROOT / "notebooks/feature_checks"
PANEL_REPORTS = ROOT / "notebooks/panel_checks"
PROCESSED = ROOT / "data/processed"
MODEL_INPUTS = PROCESSED / "model_inputs"
for folder in [FEATURE_REPORTS, PANEL_REPORTS, MODEL_INPUTS]:
    folder.mkdir(parents=True, exist_ok=True)
SECTORS = ["XLB", "XLE", "XLF", "XLI", "XLK", "XLP", "XLU", "XLV", "XLY"]
ETFS = SECTORS + ["SPY", "TLT", "IEF", "GLD", "HYG", "LQD"]
AUX = ["TLT", "IEF", "GLD", "HYG", "LQD"]
NY = "America/New_York"
FEATURE_VERSION = "03-features-v2"
PANEL_VERSION = "04-panels-v1"
SCHEMES = ["S0", "S1", "S2"]
KEYS = ["date", "ticker"]
TIME_FIELDS = ["signal_at", "entry_at", "label_available_at"]
read_options = dict(parse_dates=["date"], dtype={"ticker":"str"},
                    low_memory=False, float_precision="round_trip")
XLF_EVENT = pd.Timestamp("2016-09-19")
pd.set_option("display.max_columns", 12)
pd.set_option("display.float_format", lambda x: f"{x:.7g}")

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()


def check(name, passed, detail):
    validation_records.append(dict(check=name, passed=bool(passed), detail=str(detail)))
    if not passed:
        raise AssertionError(f"{name}: {detail}")

def save_csv(frame, relative_path, records, include_bytes=False):
    out = frame.copy()
    for field in out:
        if isinstance(out[field].dtype, pd.DatetimeTZDtype):
            out[field] = out[field].astype(str).replace("NaT", None)
        elif pd.api.types.is_datetime64_any_dtype(out[field]):
            out[field] = out[field].dt.strftime("%Y-%m-%d")
    relative_path = str(relative_path).replace("\\", "/")
    path = ROOT / relative_path
    out.to_csv(path, index=False, encoding="utf-8", float_format="%.17g")
    record = dict(file=relative_path, rows=len(frame), columns=len(frame.columns))
    if include_bytes:
        record["bytes"] = path.stat().st_size
    record["sha256"] = sha256(path)
    records.append(record)

cleaning_manifest_path = ROOT / "notebooks/cleaning_checks/cleaning_manifest.json"
cleaning_manifest_hash = sha256(cleaning_manifest_path)
cleaning_manifest = json.loads(cleaning_manifest_path.read_text(encoding='utf-8'))
source_names = ["etf_prices_aligned.csv", "etf_base_returns.csv", "vix_aligned.csv",
                "dgs3mo_observations.csv", "risk_free_open_intervals.csv"]
cleaning_hashes = {}
for record in cleaning_manifest["output_files"]:
    relative_path = record["file"].replace("\\", "/")
    if relative_path in cleaning_hashes:
        raise ValueError(f"Duplicate cleaned output in manifest: {relative_path}")
    cleaning_hashes[relative_path] = record["sha256"]
source_hashes = {}
for name in source_names:
    rel = f"data/interim/{name}"
    if rel not in cleaning_hashes:
        raise ValueError(f"Missing cleaned output in manifest: {rel}. Rerun notebook 01.")
    expected = cleaning_hashes[rel]
    source_hashes[rel] = sha256(ROOT/rel)
    assert source_hashes[rel] == expected, f"Cleaned input checksum mismatch: {name}"

prices = pd.read_csv(INTERIM/"etf_prices_aligned.csv", parse_dates=["date"],
                     dtype={"ticker":"string", "volume":"Int64"},
                     low_memory=False, float_precision="round_trip")
returns = pd.read_csv(INTERIM/"etf_base_returns.csv", parse_dates=["date", "previous_date"],
                      dtype={"ticker":"string"}, low_memory=False, float_precision="round_trip")
vix = pd.read_csv(INTERIM/"vix_aligned.csv", parse_dates=["date"],
                  low_memory=False, float_precision="round_trip").set_index("date")
price_by_asset = {t: prices.loc[prices["ticker"].eq(t)].set_index("date").sort_index()
                  for t in ETFS}
return_by_asset = {t: returns.loc[returns["ticker"].eq(t)].set_index("date").sort_index()
                   for t in ETFS}
calendar = price_by_asset["SPY"].index
assert calendar.is_unique and calendar.is_monotonic_increasing and len(calendar) == 5218
assert all(p.index.equals(calendar) for p in price_by_asset.values())
assert all(r.index.equals(calendar) for r in return_by_asset.values())
assert vix.index.equals(calendar)

## Feature Construction

Rolling statistics require complete windows. Cumulative returns compound cash daily returns; 12–1 momentum uses $t-251$ through $t-21$. Volatility is daily (`ddof=1`). No fitted preprocessing is applied.

### Historical Returns


In [ ]:
def cumulative_return(r, length, skip_recent=0):
    if not r.dropna().gt(-1).all():
        raise ValueError("Cash returns must exceed -100%.")
    width = length - skip_recent
    log_return = np.log1p(r).shift(skip_recent)
    return np.expm1(log_return.rolling(width, min_periods=width).sum())

def build_histories(return_inputs):
    histories = {}
    for ticker, data in return_inputs.items():
        r = data["ret_cc_cash"].where(data["ret_cc_cash_valid"])
        h = pd.DataFrame(index=data.index)
        for lag in [0, 1, 2]:
            h[f"ret_1d_lag{lag}"] = r.shift(lag)
        for length in [5, 20, 60]:
            h[f"cumret_{length}d"] = cumulative_return(r, length)
        h["mom_12_1"] = cumulative_return(r, 252, 21)
        for length in [20, 60]:
            h[f"vol_{length}d"] = r.rolling(length, min_periods=length).std(ddof=1)
        histories[ticker] = h
    return histories

histories = build_histories(return_by_asset)


### S0: Common Market State

SPY, VIX and five auxiliary ETFs provide 34 common features, including four cross-asset return spreads.


In [ ]:
def build_common_features(history, vix_inputs):
    out = pd.DataFrame(index=history["SPY"].index)
    for name in ["ret_1d_lag0", "ret_1d_lag1", "ret_1d_lag2",
                 "cumret_5d", "cumret_20d", "cumret_60d", "vol_20d"]:
        out[f"mkt_SPY_{name}"] = history["SPY"][name]
    v = vix_inputs["vix_close"].where(vix_inputs["vix_close_valid"])
    out["state_VIX_log_level"] = np.log(v.where(v > 0))
    for lag in [1, 5]:
        out[f"state_VIX_pct_change_{lag}d"] = v / v.shift(lag) - 1
    for ticker in AUX:
        out[f"aux_{ticker}_cumret_1d"] = history[ticker]["ret_1d_lag0"]
        for length in [5, 20]:
            out[f"aux_{ticker}_cumret_{length}d"] = history[ticker][f"cumret_{length}d"]
        out[f"aux_{ticker}_vol_20d"] = history[ticker]["vol_20d"]
    for a, b in [("TLT", "IEF"), ("HYG", "LQD")]:
        for length in [5, 20]:
            out[f"state_{a}_minus_{b}_cumret_{length}d"] = (
                history[a][f"cumret_{length}d"] - history[b][f"cumret_{length}d"])
    return out

common_features = build_common_features(histories, vix)


### S0: Sector Features

Sector history, relative returns, activity and risk provide 23 features. Dollar volume uses provider units; Amihud is scaled by $10^6$. Formulas and sources are recorded in the catalog.


In [ ]:
def build_own_features(price_inputs, return_inputs, history):
    own = {}
    spy_r = return_inputs["SPY"]["ret_cc_cash"]
    spy_var60 = spy_r.rolling(60, min_periods=60).var(ddof=1)
    for ticker in SECTORS:
        p, r, h = price_inputs[ticker], return_inputs[ticker], history[ticker]
        out = pd.DataFrame(index=p.index)
        for name in ["ret_1d_lag0", "ret_1d_lag1", "ret_1d_lag2",
                     "cumret_5d", "cumret_20d", "cumret_60d", "mom_12_1"]:
            out[f"own_{name}"] = h[name]
        out["own_excess_SPY_1d"] = h["ret_1d_lag0"] - history["SPY"]["ret_1d_lag0"]
        for length in [5, 20, 60]:
            out[f"own_excess_SPY_{length}d"] = (
                h[f"cumret_{length}d"] - history["SPY"][f"cumret_{length}d"])
        for field, short in [("ret_overnight_cash", "overnight"), ("ret_intraday_price", "intraday")]:
            value = r[field].where(r[f"{field}_valid"])
            out[f"own_{short}_1d"] = value
            out[f"own_{short}_mean_5d"] = value.rolling(5, min_periods=5).mean()
        for length in [20, 60]:
            out[f"own_vol_{length}d"] = h[f"vol_{length}d"]
        range_squared = np.log(p["high"] / p["low"]).pow(2)
        out["own_parkinson_20d"] = np.sqrt(
            range_squared.rolling(20, min_periods=20).mean() / (4*np.log(2)))
        volume = p["volume"].astype("float64")
        preceding_volume = volume.shift(1).rolling(20, min_periods=20).mean()
        out["own_relative_volume_20d"] = volume / preceding_volume.where(preceding_volume > 0)
        dollar_volume = p["close"] * volume
        mean_dv = dollar_volume.rolling(20, min_periods=20).mean()
        out["own_log_dollar_volume_20d"] = np.log(mean_dv.where(mean_dv > 0))
        daily_illiq = h["ret_1d_lag0"].abs() / dollar_volume.where(dollar_volume > 0)
        out["own_amihud_20d_x1e6"] = daily_illiq.rolling(20, min_periods=20).mean() * 1e6
        covariance = h["ret_1d_lag0"].rolling(60, min_periods=60).cov(spy_r, ddof=1)
        out["own_beta_SPY_60d"] = covariance / spy_var60.where(spy_var60 > 0)
        out["own_corr_SPY_60d"] = h["ret_1d_lag0"].rolling(60, min_periods=60).corr(spy_r)
        own[ticker] = out
    return own

own_features = build_own_features(price_by_asset, return_by_asset, histories)


### S1: Cross-Sector Returns

Use three daily returns and 5/20-day cumulative returns from the other eight sectors, with fixed ticker names.


In [ ]:
CROSS_SUFFIXES = ["ret_1d_lag0", "ret_1d_lag1", "ret_1d_lag2", "cumret_5d", "cumret_20d"]
def build_cross_features(history):
    return pd.DataFrame({
        f"cross_{ticker}_{name}": history[ticker][name]
        for ticker in SECTORS for name in CROSS_SUFFIXES})

cross_features = build_cross_features(histories)


### S2: Sector Relationships

Ranks and dispersion require all nine sectors. Signed 60-day correlations require all eight target pairs or all 36 sector pairs.


In [ ]:
def build_relation_features(history):
    r1 = pd.DataFrame({t: history[t]["ret_1d_lag0"] for t in SECTORS})
    r20 = pd.DataFrame({t: history[t]["cumret_20d"] for t in SECTORS})
    complete1, complete20 = r1.notna().all(axis=1), r20.notna().all(axis=1)
    rank20 = (r20.rank(axis=1, method="average") - 1) / 8
    rank20.loc[~complete20, :] = np.nan
    dispersion1 = r1.std(axis=1, ddof=1).where(complete1)
    dispersion20 = r20.std(axis=1, ddof=1).where(complete20)
    pair_corr = {
        (a, b): r1[a].rolling(60, min_periods=60).corr(r1[b])
        for a, b in combinations(SECTORS, 2)}
    all_corr = pd.DataFrame({f"{a}_{b}": c for (a,b),c in pair_corr.items()})
    system_mean = all_corr.mean(axis=1, skipna=False)
    result = {}
    for ticker in SECTORS:
        own_pairs = pd.DataFrame({f"{a}_{b}": c
                                 for (a,b),c in pair_corr.items() if ticker in (a,b)})
        result[ticker] = pd.DataFrame({
            "rel_rank20_9sectors": rank20[ticker],
            "rel_dispersion1_9sectors": dispersion1,
            "rel_dispersion20_9sectors": dispersion20,
            "rel_own_mean_corr60_other8": own_pairs.mean(axis=1, skipna=False),
            "rel_market_mean_corr60_all36": system_mean})
    return result

relation_features = build_relation_features(histories)
own_columns = list(own_features["XLB"].columns)
common_columns = list(common_features.columns)
relation_columns = list(relation_features["XLB"].columns)
feature_sets = {}
features_by_target = {}
for ticker in SECTORS:
    s0 = own_columns + common_columns
    s1 = s0 + [c for c in cross_features if not c.startswith(f"cross_{ticker}_")]
    s2 = s1 + relation_columns
    feature_sets[ticker] = {"S0":s0, "S1":s1, "S2":s2}
    features_by_target[ticker] = pd.concat(
        [own_features[ticker], common_features, cross_features, relation_features[ticker]], axis=1)
feature_table = pd.concat([
    f.assign(ticker=t).reset_index()
    for t,f in features_by_target.items()], ignore_index=True
).sort_values(["date","ticker"]).reset_index(drop=True)


## Targets and Information Timing

Signals are dated at 18:00 New York time. The target is sector minus SPY cash return from the next open to the following open. Labels become available at the exit open.

In [ ]:
def split_for_dates(dates):
    values = pd.Series("warmup", index=dates, dtype="str")
    values.loc[(dates >= "2008-01-01") & (dates <= "2017-12-31")] = "train"
    values.loc[(dates >= "2018-01-01") & (dates <= "2022-12-31")] = "validation"
    values.loc[dates >= "2023-01-01"] = "test"
    return values

split_by_date = split_for_dates(calendar)
split_end = {"warmup":pd.Timestamp("2007-12-31"), "train":pd.Timestamp("2017-12-31"),
             "validation":pd.Timestamp("2022-12-31"), "test":pd.Timestamp("2026-09-30")}
date_series = pd.Series(calendar, index=calendar)
entry_dates, exit_dates = date_series.shift(-1), date_series.shift(-2)
label_parts = []
spy_forward = return_by_asset["SPY"]["ret_oo_cash"].shift(-2)
for ticker in SECTORS:
    forward = return_by_asset[ticker]["ret_oo_cash"].shift(-2)
    y = pd.DataFrame(index=calendar)
    y["ticker"] = ticker
    y["split"] = split_by_date
    y["signal_at"] = (date_series + pd.Timedelta(hours=18)).dt.tz_localize(NY).astype(str)
    y["entry_date"], y["exit_date"] = entry_dates, exit_dates
    y["entry_at"] = (entry_dates + pd.Timedelta(hours=9,minutes=30)).dt.tz_localize(NY)
    exit_timestamp = (exit_dates + pd.Timedelta(hours=9,minutes=30)).dt.tz_localize(NY)
    y["label_available_at"] = exit_timestamp
    y["sector_forward_cash_return"], y["spy_forward_cash_return"] = forward, spy_forward
    y["target_relative_return"] = forward - spy_forward
    y["label_valid"] = y["target_relative_return"].notna()
    y["label_within_split"] = y["label_valid"] & (
        y["exit_date"] <= y["split"].map(split_end))
    y["invalid_reason"] = ""
    y.loc[entry_dates.isna() | exit_dates.isna(), "invalid_reason"] = "future_open_outside_data_cutoff"
    if ticker == "XLF":
        y.loc[exit_dates.eq(XLF_EVENT), "invalid_reason"] = "xlf_in_kind_distribution"
    y.loc[~y["label_valid"] & y["invalid_reason"].eq(""), "invalid_reason"] = "missing_holding_endpoint"
    label_parts.append(y.reset_index())
label_table = pd.concat(label_parts, ignore_index=True).sort_values(
    ["date","ticker"]).reset_index(drop=True)


## Sample Eligibility

Validity uses target-specific predictor lists. Common masks identify the S0/S1/S2 intersection; training additionally requires mature labels. Missing-feature details are recorded once.

In [ ]:
reason_definitions = [
    ("OK", "none", "No validity restriction"),
    ("H", "feature", "Insufficient initial coverage or lookback history"),
    ("XO", "feature", "XLF own-sector window includes the unverified distribution event"),
    ("XC", "feature", "Cross-sector XLF window includes the unverified distribution event"),
    ("XR", "feature", "Sector rank, dispersion or correlation depends on the XLF event"),
    ("M", "feature", "Unexplained missing predictor; review required"),
    ("LX", "label", "Holding-period label affected by the XLF distribution event"),
    ("LE", "label", "Holding-period endpoint is beyond the data cutoff"),
    ("LM", "label", "Other missing label; review required"),
    ("B", "scoring", "Valid label crossing an evaluation boundary"),
    ("W", "scoring", "2006-2007 warmup observations excluded from evaluation"),
]
reason_dictionary = pd.DataFrame(reason_definitions, columns=["code", "scope", "definition"])
reason_order = [code for code, _, _ in reason_definitions if code != "OK"]
event_pos = calendar.get_loc(XLF_EVENT)

def xlf_missing_reason(ticker, feature, date):
    offset = calendar.get_loc(date) - event_pos
    code, start, stop = None, 0, -1
    if feature.startswith("cross_XLF_") and ticker != "XLF":
        metric = feature.removeprefix("cross_XLF_")
        if metric.startswith("ret_1d_lag"):
            start = stop = int(metric[-1])
        else:
            stop = int(metric.split("_")[1][:-1]) - 1
        code = "XC"
    elif feature in relation_columns:
        stop = (59 if "corr60" in feature else
                19 if feature in ["rel_rank20_9sectors", "rel_dispersion20_9sectors"] else 0)
        code = "XR"
    elif ticker == "XLF" and feature.startswith("own_"):
        metric = feature.removeprefix("own_")
        if metric.startswith("ret_1d_lag"):
            start = stop = int(metric[-1])
        elif metric == "mom_12_1":
            start, stop = 21, 251
        elif metric.startswith(("cumret_", "vol_")):
            stop = int(metric.split("_")[1][:-1]) - 1
        elif metric.startswith("excess_SPY_"):
            stop = int(metric.split("_")[-1][:-1]) - 1
        elif metric == "overnight_1d":
            stop = 0
        elif metric == "overnight_mean_5d":
            stop = 4
        elif metric == "amihud_20d_x1e6":
            stop = 19
        elif metric in ["beta_SPY_60d", "corr_SPY_60d"]:
            stop = 59
        code = "XO"
    return code if code is not None and start <= offset <= stop else "M"

missing_rows, validity_parts = [], []
for ticker, features in features_by_target.items():
    v = pd.DataFrame(index=calendar)
    v["ticker"], v["split"] = ticker, split_by_date
    labels = label_table.loc[label_table["ticker"].eq(ticker)].set_index("date")
    v["label_valid"], v["label_within_split"] = labels["label_valid"], labels["label_within_split"]
    row_codes = {date: set() for date in calendar}

    for feature in feature_sets[ticker]["S2"]:
        values = features[feature]
        first = values.first_valid_index()
        for date in values.index[values.isna()]:
            code = "H" if first is not None and date < first else xlf_missing_reason(ticker, feature, date)
            missing_rows.append(dict(date=date, ticker=ticker, feature=feature, reason_code=code))
            row_codes[date].add(code)
    for scheme in ["S0", "S1", "S2"]:
        key = scheme.lower()
        valid = features[feature_sets[ticker][scheme]].notna().all(axis=1)
        v[f"features_valid_{key}"] = valid
        v[f"scoreable_{key}"] = valid & v["label_within_split"] & v["split"].ne("warmup")
    v["common_features_valid"] = v[[f"features_valid_{s}" for s in ["s0", "s1", "s2"]]].all(axis=1)
    v["common_scoreable"] = v[[f"scoreable_{s}" for s in ["s0", "s1", "s2"]]].all(axis=1)
    for date in calendar:
        if not labels.at[date, "label_valid"]:
            label_reason = labels.at[date, "invalid_reason"]
            code = {"xlf_in_kind_distribution":"LX", "future_open_outside_data_cutoff":"LE"}.get(label_reason, "LM")
            row_codes[date].add(code)
        elif not labels.at[date, "label_within_split"]:
            row_codes[date].add("B")
        if split_by_date.at[date] == "warmup":
            row_codes[date].add("W")
    v["reason_code"] = ["+".join(c for c in reason_order if c in row_codes[d]) or "OK" for d in calendar]
    validity_parts.append(v.reset_index())

feature_validity = pd.concat(validity_parts, ignore_index=True).sort_values(
    ["date", "ticker"]).reset_index(drop=True)
missing_feature_details = pd.DataFrame(missing_rows).sort_values(
    ["date", "ticker", "feature"]).reset_index(drop=True)
count_rows, common_rows = [], []
for (ticker, period), group in feature_validity.groupby(["ticker", "split"], sort=False):
    for scheme in ["S0", "S1", "S2"]:
        key = scheme.lower()
        count_rows.append(dict(
            ticker=ticker, split=period, feature_set=scheme, calendar_rows=len(group),
            feature_count=len(feature_sets[ticker][scheme]),
            feature_valid_rows=int(group[f"features_valid_{key}"].sum()),
            feature_and_label_valid_rows=int((group[f"features_valid_{key}"] & group["label_valid"]).sum()),
            scoreable_rows=int(group[f"scoreable_{key}"].sum())))
    common_rows.append(dict(
        ticker=ticker, split=period, calendar_rows=len(group),
        common_features_valid_rows=int(group["common_features_valid"].sum()),
        common_feature_and_label_valid_rows=int((group["common_features_valid"] & group["label_valid"]).sum()),
        common_scoreable_rows=int(group["common_scoreable"].sum())))
sample_counts = pd.DataFrame(count_rows)
common_sample_counts = pd.DataFrame(common_rows)


### Feature Catalog

In [ ]:
catalog_rows = []
def register(name, group, scheme, assets, fields, formula, window, units, why, reference="project_definition"):
    catalog_rows.append(dict(
        feature=name, group=group, first_in_set=scheme, source_assets=assets,
        source_fields=fields, formula=formula, lookback_trading_days=window,
        units=units, available_at="signal_date 18:00 America/New_York",
        purpose=why, reference=reference))

refs = {
    "momentum":"https://doi.org/10.1111/j.1540-6261.1993.tb04702.x",
    "overnight":"https://doi.org/10.1016/j.jfineco.2019.03.011",
    "parkinson":"https://doi.org/10.1086/296071",
    "volume":"https://doi.org/10.1111/0022-1082.00349",
    "amihud":"https://doi.org/10.1016/S1386-4181(01)00024-6",
    "industry":"https://www.sciencedirect.com/science/article/pii/S0304405X06001383"}
for lag in [0,1,2]:
    register(f"own_ret_1d_lag{lag}","own_history","S0","target","ret_cc_cash",
             f"r[t-{lag}]",lag+1,"decimal_return","Short-term continuation or reversal")
for k in [5,20,60]:
    register(f"own_cumret_{k}d","own_history","S0","target","ret_cc_cash",
             f"prod(1+r[t-{k-1}:t])-1; all {k} returns valid",k,"decimal_return","Short- and medium-horizon trends")
register("own_mom_12_1","own_history","S0","target","ret_cc_cash",
         "prod(1+r[t-251:t-21])-1; 231 complete returns",252,"decimal_return","Longer-horizon momentum",refs["momentum"])
for k in [1,5,20,60]:
    register(f"own_excess_SPY_{k}d","own_market_relative","S0","target,SPY","ret_cc_cash",
             f"R_target_{k}d - R_SPY_{k}d",k,"decimal_return","Market-relative performance")
for part, field in [("overnight","ret_overnight_cash"),("intraday","ret_intraday_price")]:
    register(f"own_{part}_1d","own_trading_session","S0","target",field,
             f"{field}[t]",1,"decimal_return","Trading-session information",refs["overnight"])
    register(f"own_{part}_mean_5d","own_trading_session","S0","target",field,
             f"mean({field}[t-4:t]); min_periods=5",5,"decimal_return","Trading-session trends",refs["overnight"])
for k in [20,60]:
    register(f"own_vol_{k}d","own_risk","S0","target","ret_cc_cash",
             f"std(r[t-{k-1}:t], ddof=1); complete window",k,"daily_volatility","Historical risk")
register("own_parkinson_20d","own_risk","S0","target","high,low",
         "sqrt(mean20(log(high/low)^2)/(4*log(2)))",20,"daily_volatility","High-low volatility estimate",refs["parkinson"])
register("own_relative_volume_20d","own_activity","S0","target","volume",
         "volume[t]/mean(volume[t-20:t-1]); excludes current day",21,"ratio","Trading activity; not fund flows",refs["volume"])
register("own_log_dollar_volume_20d","own_activity","S0","target","close,volume",
         "log(mean20(close*volume)); provider-unit proxy",20,"log_dollar_volume_proxy","Trading-size proxy",refs["volume"])
register("own_amihud_20d_x1e6","own_liquidity","S0","target","ret_cc_cash,close,volume",
         "1e6*mean20(abs(r)/(close*volume)); positive denominator",20,"scaled_illiquidity_proxy","Illiquidity proxy; not realized costs",refs["amihud"])
register("own_beta_SPY_60d","own_market_exposure","S0","target,SPY","ret_cc_cash",
         "cov60(r_target,r_SPY,ddof=1)/var60(r_SPY,ddof=1)",60,"beta","Market exposure")
register("own_corr_SPY_60d","own_market_exposure","S0","target,SPY","ret_cc_cash",
         "corr60(r_target,r_SPY); all 60 pairs valid",60,"correlation","Market co-movement")
for name in common_columns:
    if name.startswith("mkt_SPY_"):
        metric = name.removeprefix("mkt_SPY_")
        formula = metric+" of SPY; same complete-window convention as own features"
        window = int(metric.split("_")[1][:-1]) if metric.startswith(("cumret_","vol_")) else int(metric[-1])+1
        assets, fields, units = "SPY","ret_cc_cash","daily_volatility" if "vol" in metric else "decimal_return"
    elif name == "state_VIX_log_level":
        formula,window,assets,fields,units = "log(VIX_Close[t])",1,"VIX","vix_close","log_index_level"
    elif name.startswith("state_VIX_pct_change_"):
        k = int(name.split("_")[-1][:-1])
        formula,window,assets,fields,units = f"VIX[t]/VIX[t-{k}]-1",k+1,"VIX","vix_close","decimal_change"
    elif name.startswith("aux_"):
        _,assets,*metric_parts = name.split("_")
        metric = "_".join(metric_parts)
        window = int(metric.split("_")[1][:-1])
        formula,fields,units = metric+"; complete cash-return window","ret_cc_cash",(
            "daily_volatility" if metric.startswith("vol") else "decimal_return")
    else:
        _,a,_,b,_,kd = name.split("_")
        window,assets,fields,units = int(kd[:-1]),f"{a},{b}","ret_cc_cash","decimal_return_difference"
        formula = f"R_{a}_{kd}-R_{b}_{kd}; not a yield or credit spread"
    register(name,"common_state","S0",assets,fields,formula,window,units,"Common market and cross-asset controls")
for ticker in SECTORS:
    for metric in CROSS_SUFFIXES:
        window = int(metric[-1])+1 if metric.startswith("ret_") else int(metric.split("_")[1][:-1])
        register(f"cross_{ticker}_{metric}","cross_sector","S1",ticker,"ret_cc_cash",
                 metric+"; include only when source ticker != target ticker",window,"decimal_return",
                 "Incremental cross-sector history",refs["industry"])
for name,formula,window,units in [
    ("rel_rank20_9sectors","(average_rank_across_9(R20)-1)/8; all 9 required",20,"rank_0_to_1"),
    ("rel_dispersion1_9sectors","std_across_9(r1,ddof=1); all 9 required",1,"cross_section_std"),
    ("rel_dispersion20_9sectors","std_across_9(R20,ddof=1); all 9 required",20,"cross_section_std"),
    ("rel_own_mean_corr60_other8","signed mean of all 8 target-other corr60; all required",60,"correlation"),
    ("rel_market_mean_corr60_all36","signed mean of all 36 unique pair corr60; all required",60,"correlation")]:
    register(name,"sector_relations","S2","all_9_sectors","ret_cc_cash",formula,window,units,"Sector position and co-movement")
feature_catalog = pd.DataFrame(catalog_rows)
assert feature_catalog["feature"].is_unique
assert set(feature_catalog["feature"]) == set(features_by_target["XLB"].columns)
feature_catalog = feature_catalog.set_index("feature").loc[
    list(features_by_target["XLB"].columns)].rename_axis("feature").reset_index()
feature_catalog["relationship_type"] = "none_flagged"
feature_catalog["related_features"] = ""
for k in [1, 5, 20, 60]:
    own = "own_ret_1d_lag0" if k == 1 else f"own_cumret_{k}d"
    market = "mkt_SPY_ret_1d_lag0" if k == 1 else f"mkt_SPY_cumret_{k}d"
    mask = feature_catalog["feature"].eq(f"own_excess_SPY_{k}d")
    feature_catalog.loc[mask, ["relationship_type", "related_features"]] = ["exact_difference", f"{own} - {market}"]
for a, b in [("TLT", "IEF"), ("HYG", "LQD")]:
    for k in [5, 20]:
        mask = feature_catalog["feature"].eq(f"state_{a}_minus_{b}_cumret_{k}d")
        feature_catalog.loc[mask, ["relationship_type", "related_features"]] = [
            "exact_difference", f"aux_{a}_cumret_{k}d - aux_{b}_cumret_{k}d"]
for name, metric in [("rel_rank20_9sectors", "cumret_20d"),
                     ("rel_dispersion1_9sectors", "ret_1d_lag0"),
                     ("rel_dispersion20_9sectors", "cumret_20d")]:
    mask = feature_catalog["feature"].eq(name)
    feature_catalog.loc[mask, ["relationship_type", "related_features"]] = [
        "same_date_summary", f"target own {metric} plus other-eight cross {metric}"]
for name in ["rel_own_mean_corr60_other8", "rel_market_mean_corr60_all36"]:
    mask = feature_catalog["feature"].eq(name)
    feature_catalog.loc[mask, ["relationship_type", "related_features"]] = [
        "longer_path_summary", "60 daily paired returns; not recoverable from current S1 summaries alone"]


### Split Diagnostics

Audit the five 2:1 splits on 2025-12-05 against four controls and the saved Yahoo JSON. Retain provider units without another split adjustment.


In [ ]:
SPLIT_DATE = pd.Timestamp("2025-12-05")
SPLIT_ETFS = ["XLB", "XLE", "XLK", "XLU", "XLY"]
CONTROL_ETFS = ["XLI", "XLP", "XLV", "SPY"]
OFFICIAL_URL = "https://investors.statestreet.com/investor-news-events/press-releases/news-details/2025/State-Street-Investment-Management-Announces-Share-Splits-for-Five-Select-Sector-SPDR-ETFs/default.aspx"
OFFICIAL_PRICE_DATE = pd.Timestamp("2025-11-18")
OFFICIAL_PRE_SPLIT_PRICES = {"XLB":85.52, "XLE":91.06, "XLK":279.03, "XLU":89.22, "XLY":224.50}
split_pos = calendar.get_loc(SPLIT_DATE)
audit_source_hashes = {}
daily_parts, window_rows, unit_rows = [], [], []
for ticker in SPLIT_ETFS + CONTROL_ETFS:
    p = price_by_asset[ticker]
    daily = p[["close", "volume", "stock_splits"]].copy()
    daily["dollar_volume"] = p["close"] * p["volume"].astype(float)

    if ticker in SECTORS:
        metrics = own_features[ticker]
    else:
        metrics = pd.DataFrame(index=calendar)
        metrics["own_relative_volume_20d"] = p["volume"].astype(float) / p["volume"].astype(float).shift(1).rolling(20, min_periods=20).mean()
        metrics["own_log_dollar_volume_20d"] = np.log(daily["dollar_volume"].rolling(20, min_periods=20).mean())
        metrics["own_amihud_20d_x1e6"] = (return_by_asset[ticker]["ret_cc_cash"].abs() / daily["dollar_volume"]).rolling(20, min_periods=20).mean()*1e6
    audit_metrics = ["own_relative_volume_20d", "own_log_dollar_volume_20d", "own_amihud_20d_x1e6"]
    daily = daily.join(metrics[audit_metrics])
    daily["ticker"], daily["event_date"] = ticker, SPLIT_DATE
    daily["relative_trading_day"] = np.arange(len(calendar))-split_pos
    daily["group"] = "split" if ticker in SPLIT_ETFS else "control"
    daily_parts.append(daily.iloc[split_pos-20:split_pos+20].reset_index())
    for width in [1, 5, 20]:
        pre = daily.iloc[split_pos-width:split_pos]
        post = daily.iloc[split_pos:split_pos+width]
        row = dict(ticker=ticker, group=daily["group"].iloc[0], event_date=SPLIT_DATE, window_trading_days=width,
                   pre_start=pre.index.min(), pre_end=pre.index.max(), post_start=post.index.min(), post_end=post.index.max())
        for field in ["volume", "dollar_volume"] + audit_metrics:
            before, after = float(pre[field].mean()), float(post[field].mean())
            row[f"pre_mean_{field}"], row[f"post_mean_{field}"] = before, after

            if field == "own_log_dollar_volume_20d":
                row["change_mean_log_dollar_volume_20d"] = after-before
            else:
                row[f"post_pre_ratio_{field}"] = after/before if before != 0 else np.nan
        window_rows.append(row)
    json_path = ROOT / f"data/raw/{ticker}_2006-01-01_2026-09-30_yahoo_chart.json"
    audit_source_hashes[json_path.relative_to(ROOT).as_posix()] = sha256(json_path)
    payload = json.loads(json_path.read_text(encoding='utf-8'))["chart"]["result"][0]
    json_dates = pd.to_datetime(payload["timestamp"], unit="s", utc=True).tz_convert(NY).normalize().tz_localize(None)
    json_volume = pd.Series(payload["indicators"]["quote"][0]["volume"], index=json_dates, dtype="Int64").reindex(calendar)
    same_volume = p["volume"].eq(json_volume).fillna(False) | (p["volume"].isna() & json_volume.isna())
    pre_volume = p.loc[p.index < SPLIT_DATE, "volume"].dropna()
    post_volume = p.loc[p.index >= SPLIT_DATE, "volume"].dropna()
    row = dict(ticker=ticker, group=daily["group"].iloc[0], event_date=SPLIT_DATE,
               source_split_factor=float(p.at[SPLIT_DATE, "stock_splits"]),
               pre_observations=len(pre_volume), post_observations=len(post_volume),
               pre_multiple_200_share=float(pre_volume.mod(200).eq(0).mean()),
               post_multiple_200_share=float(post_volume.mod(200).eq(0).mean()),
               pre_multiple_400_share=float(pre_volume.mod(400).eq(0).mean()),
               json_volume_mismatch_rows=int((~same_volume).sum()),
               official_price_date=OFFICIAL_PRICE_DATE if ticker in SPLIT_ETFS else pd.NaT,
               official_pre_split_price=OFFICIAL_PRE_SPLIT_PRICES.get(ticker, np.nan),
               yahoo_historical_close=float(p.at[OFFICIAL_PRICE_DATE, "close"]),
               official_source_url=OFFICIAL_URL if ticker in SPLIT_ETFS else "",
               decision="keep_provider_units")
    row["official_minus_2x_yahoo_price"] = row["official_pre_split_price"] - 2*row["yahoo_historical_close"]
    unit_rows.append(row)
split_window_daily = pd.concat(daily_parts, ignore_index=True).sort_values(["date", "ticker"]).reset_index(drop=True)
split_window_summary = pd.DataFrame(window_rows)
split_unit_checks = pd.DataFrame(unit_rows)


### Feature Validation

Independent window calculations, future-prefix tests and label-maturity checks validate the constructed data.

In [ ]:
validation_records, numeric_checks = [], []

check("physical_feature_count", len(feature_table.columns)-2 == 107, "107 numeric + date/ticker")
check("per_target_feature_counts", all(
    [len(feature_sets[t][s]) for s in ["S0","S1","S2"]] == [57,97,102] for t in SECTORS), "57/97/102")
check("self_cross_columns_excluded", all(
    not any(c.startswith(f"cross_{t}_") for c in feature_sets[t]["S1"]) for t in SECTORS), "Each target uses the other eight sectors")
check("no_infinite_features", not np.isinf(feature_table.drop(columns=["date","ticker"]).to_numpy()).any(), True)
check("no_targets_in_feature_lists", not any(
    "target_relative_return" in cols or "label_valid" in cols
    for sets in feature_sets.values() for cols in sets.values()), True)
check("label_count_and_terminal_boundary", int(label_table["label_valid"].sum()) == 46943
      and not label_table.loc[label_table["date"].isin(
          [pd.Timestamp("2026-09-29"),pd.Timestamp("2026-09-30")]),"label_valid"].any(), 46943)
check("xlf_bad_label_signal_date", label_table.loc[
    label_table["ticker"].eq("XLF") & label_table["invalid_reason"].eq("xlf_in_kind_distribution"),
    "date"].tolist() == [pd.Timestamp("2016-09-15")], "Signal 09-15; entry 09-16; exit 09-19")
check("feature_history_first_complete", common_features.dropna().index.min() == pd.Timestamp("2007-05-09"),
      "HYG 20-day warmup; training starts in 2008")
check("momentum_first_available", own_features["XLB"]["own_mom_12_1"].first_valid_index()
      == pd.Timestamp("2007-01-04"), True)

date = pd.Timestamp("2020-03-24")
k = calendar.get_loc(date)
r = return_by_asset["XLB"]["ret_cc_cash"].to_numpy()
spy = return_by_asset["SPY"]["ret_cc_cash"].to_numpy()
p = price_by_asset["XLB"]
volume = p["volume"].to_numpy(dtype=float,na_value=np.nan)
dollar = p["close"].to_numpy()*volume
manual = {
    "own_cumret_5d":np.prod(1+r[k-4:k+1])-1,
    "own_cumret_20d":np.prod(1+r[k-19:k+1])-1,
    "own_mom_12_1":np.prod(1+r[k-251:k-20])-1,
    "own_vol_20d":np.std(r[k-19:k+1],ddof=1),
    "own_parkinson_20d":np.sqrt(np.mean(
        np.log(p["high"].to_numpy()[k-19:k+1]/p["low"].to_numpy()[k-19:k+1])**2)/(4*np.log(2))),
    "own_relative_volume_20d":volume[k]/np.mean(volume[k-20:k]),
    "own_log_dollar_volume_20d":np.log(np.mean(dollar[k-19:k+1])),
    "own_amihud_20d_x1e6":np.mean(np.abs(r[k-19:k+1])/dollar[k-19:k+1])*1e6,
    "own_beta_SPY_60d":np.cov(r[k-59:k+1],spy[k-59:k+1],ddof=1)[0,1]/np.var(spy[k-59:k+1],ddof=1),
    "own_corr_SPY_60d":np.corrcoef(r[k-59:k+1],spy[k-59:k+1])[0,1],
}
for feature,expected in manual.items():
    actual = features_by_target["XLB"].at[date,feature]
    passed = np.isclose(actual,expected,rtol=1e-9,atol=1e-12)
    numeric_checks.append(dict(ticker="XLB",date=date,feature=feature,
                               expected=expected,actual=actual,passed=bool(passed)))
check("independent_numpy_feature_checks", all(x["passed"] for x in numeric_checks), len(numeric_checks))
numeric_check_table = pd.DataFrame(numeric_checks)
train = sample_counts.query("split == 'train'").groupby("feature_set")["feature_valid_rows"].sum()
check("xlf_training_feature_impact_counts", train.to_dict() == {"S0":22410,"S1":22250,"S2":21930},
      train.to_dict())
non_xlf_s0 = feature_validity.query("ticker != 'XLF' and split == 'train'")
check("other_sector_S0_not_lost_to_XLF", non_xlf_s0["features_valid_s0"].all(), "8×2518")
check("XLF_momentum_recovers", pd.notna(features_by_target["XLF"].at[
    pd.Timestamp("2017-09-19"),"own_mom_12_1"]), True)
check("nine_sector_rank_bounds", feature_table["rel_rank20_9sectors"].dropna().between(0,1).all(), True)
check("cross_section_missing_not_ignored", all(
    features_by_target[t].at[XLF_EVENT,"rel_dispersion1_9sectors"] is np.nan
    or pd.isna(features_by_target[t].at[XLF_EVENT,"rel_dispersion1_9sectors"]) for t in SECTORS), True)


In [ ]:
def recompute_features_to(cutoff):
    p = {t:d.loc[:cutoff] for t,d in price_by_asset.items()}
    r = {t:d.loc[:cutoff] for t,d in return_by_asset.items()}
    h = build_histories(r)
    common = build_common_features(h, vix.loc[:cutoff])
    own = build_own_features(p,r,h)
    cross = build_cross_features(h)
    relations = build_relation_features(h)
    return {t:pd.concat([own[t],common,cross,relations[t]],axis=1) for t in SECTORS}

causality_rows = []
for cutoff in ["2016-09-15","2016-10-18","2020-03-20","2026-06-30"]:
    prefix = recompute_features_to(cutoff)
    for ticker in SECTORS:
        expected = features_by_target[ticker].loc[:cutoff]
        actual = prefix[ticker]
        a,b = expected.to_numpy(),actual.to_numpy()
        same = np.isclose(a,b,rtol=1e-10,atol=1e-12,equal_nan=True)
        finite = np.isfinite(a) & np.isfinite(b)
        causality_rows.append(dict(
            cutoff=pd.Timestamp(cutoff),ticker=ticker,rows=len(actual),
            compared_cells=int(a.size),passed=bool(same.all()),
            max_abs_difference=float(np.abs(a[finite]-b[finite]).max()) if finite.any() else 0))
causality_checks = pd.DataFrame(causality_rows)
check("prefix_causality_all_features", causality_checks["passed"].all(),
      f"{len(causality_checks)} sector/cutoff comparisons")

boundary_rows = []
for cutoff in ["2017-12-29","2022-12-30"]:
    fit_at = (pd.Timestamp(cutoff)+pd.Timedelta(hours=18)).tz_localize(NY)
    observed = label_table.loc[(label_table["date"]>=pd.Timestamp("2008-01-01"))
        & label_table["label_valid"] & (label_table["label_available_at"]<=fit_at)]
    latest = observed["date"].max()
    boundary_rows.append(dict(fit_date=pd.Timestamp(cutoff),latest_mature_signal=latest))
boundary_checks = pd.DataFrame(boundary_rows)
check("training_label_maturity_boundaries",
      boundary_checks["latest_mature_signal"].tolist()
      == [pd.Timestamp("2017-12-27"),pd.Timestamp("2022-12-28")], boundary_checks.to_dict("records"))
check("validation_does_not_use_test_returns", not label_table.loc[
    label_table["split"].eq("validation")
    & (label_table["exit_date"]>pd.Timestamp("2022-12-31")),"label_within_split"].any(), True)
check("source_hashes_unchanged", all(
    sha256(ROOT/p)==h for p,h in source_hashes.items()), len(source_hashes))
check("cleaning_manifest_unchanged", sha256(cleaning_manifest_path)==cleaning_manifest_hash, True)

detail_keys = ["date", "ticker", "feature"]
check("missing_details_unique", not missing_feature_details.duplicated(detail_keys).any(), len(missing_feature_details))
reconstructed_ok, actual_missing_ok = True, True
for ticker in SECTORS:
    detail = missing_feature_details.loc[missing_feature_details["ticker"].eq(ticker)]
    selected = features_by_target[ticker][feature_sets[ticker]["S2"]]
    actual_keys = {(date, feature) for feature in selected for date in selected.index[selected[feature].isna()]}
    detail_key_set = set(zip(detail["date"], detail["feature"]))
    actual_missing_ok &= actual_keys == detail_key_set
    main = feature_validity.loc[feature_validity["ticker"].eq(ticker)].set_index("date")
    for scheme in ["S0", "S1", "S2"]:
        bad_dates = detail.loc[detail["feature"].isin(feature_sets[ticker][scheme]), "date"]
        reconstructed_ok &= np.array_equal(~calendar.isin(bad_dates), main[f"features_valid_{scheme.lower()}"].to_numpy())
check("missing_details_exactly_match_selected_NaNs", actual_missing_ok, "Actual target-specific S2 missing values; self-cross columns excluded")
check("validity_reconstructed_from_long_audit", reconstructed_ok, "Validity reconstructed from missing-feature details")
check("missing_details_all_explained", not missing_feature_details["reason_code"].eq("M").any(),
      missing_feature_details["reason_code"].value_counts().to_dict())
check("lean_validity_schema", len(feature_validity.columns) == 14 and not any(
      c.startswith(("invalid_features_", "invalid_reason_")) for c in feature_validity), list(feature_validity.columns))
feature_flags = [f"features_valid_{s}" for s in ["s0", "s1", "s2"]]
score_flags = [f"scoreable_{s}" for s in ["s0", "s1", "s2"]]
check("common_masks_are_three_way_intersections",
      feature_validity["common_features_valid"].equals(feature_validity[feature_flags].all(axis=1))
      and feature_validity["common_scoreable"].equals(feature_validity[score_flags].all(axis=1)), True)
common_counts = common_sample_counts.groupby("split")[["common_features_valid_rows", "common_scoreable_rows"]].sum()
check("common_sample_counts", common_counts.to_dict("index") == {
    "warmup":{"common_features_valid_rows":1476, "common_scoreable_rows":0},
    "train":{"common_features_valid_rows":21930, "common_scoreable_rows":21911},
    "validation":{"common_features_valid_rows":11331, "common_scoreable_rows":11313},
    "test":{"common_features_valid_rows":8451, "common_scoreable_rows":8433}}, common_counts.to_dict("index"))
check("all_main_tables_sorted_unique_same_keys", all(
    not table.duplicated(["date", "ticker"]).any()
    and table[["date", "ticker"]].equals(feature_table[["date", "ticker"]])
    for table in [feature_table, label_table, feature_validity]), True)
late_rows = feature_validity["date"].isin(pd.to_datetime(["2026-09-29", "2026-09-30"]))
check("common_prediction_and_scoring_are_distinct", feature_validity.loc[late_rows, "common_features_valid"].all()
      and not feature_validity.loc[late_rows, "common_scoreable"].any(), "Last two dates have predictors but no evaluable labels")
check("split_json_volume_matches_existing_tables", split_unit_checks["json_volume_mismatch_rows"].eq(0).all(), "Same-provider comparison, not independent vendor validation")
split_rows = split_unit_checks.loc[split_unit_checks["group"].eq("split")]
check("split_official_date_factor_and_price", split_rows["source_split_factor"].eq(2).all()
      and split_rows["official_minus_2x_yahoo_price"].abs().le(1e-4).all(),
      "Official 2025-12-05 2:1 split; 11/18 prices agree within floating-point tolerance")
check("split_diagnostic_windows_complete", len(split_window_daily) == 360 and len(split_window_summary) == 27
      and not split_window_daily[["close", "volume", "dollar_volume"]].isna().any().any(), "Nine assets, 40 dates and three windows per asset")
check("additional_audit_sources_unchanged", all(sha256(ROOT/p) == h for p, h in audit_source_hashes.items()), len(audit_source_hashes))

token_counts = {code: int(feature_validity["reason_code"].str.split("+").map(lambda parts: code in parts).sum())
                for code in ["H", "XO", "XC", "XR", "M", "LX", "LE", "LM", "B", "W"]}
check("reason_counts_match_known_dependencies", token_counts == {
    "H":3042, "XO":252, "XC":160, "XR":540, "M":0, "LX":1, "LE":18, "LM":0, "B":54, "W":4518}, token_counts)

feature_validation_checks = pd.DataFrame(validation_records)


### Feature Exports

In [ ]:
feature_output_records = []

feature_path = INTERIM/"sector_features.csv"
save_csv(feature_table,"data/interim/sector_features.csv", feature_output_records)
reloaded = pd.read_csv(
    feature_path, parse_dates=["date"], dtype={"ticker":"str"},
    low_memory=False, float_precision="round_trip")

pd.testing.assert_frame_equal(feature_table,reloaded,check_exact=True)
save_csv(label_table,"data/interim/prediction_targets.csv", feature_output_records)
save_csv(feature_validity,"data/interim/feature_validity.csv", feature_output_records)
feature_spec = dict(version=FEATURE_VERSION,model_structure="one_model_per_target_sector",by_target=feature_sets,
            physical_feature_columns=list(features_by_target["XLB"].columns),
            excluded_from_predictors=sorted(set(label_table.columns) | set(feature_validity.columns) | {"DGS3MO"}),
            counts={"S0":57,"S1":97,"S2":102},
            main_comparison_masks={"prediction_or_training":"common_features_valid", "strict_scoring":"common_scoreable"},
            final_panel_policy="Sector-specific inputs contain the selected predictors with fixed ticker names and exclude self cross columns.",
            note="The union table retains named cross-sector columns; by_target excludes the target sector.")
spec_path = FEATURE_REPORTS/"feature_sets.json"
spec_path.write_text(json.dumps(feature_spec,ensure_ascii=False,indent=2),encoding="utf-8")
feature_output_records.append(dict(file=spec_path.relative_to(ROOT).as_posix(),rows=None,columns=None,sha256=sha256(spec_path)))

feature_diagnostics = []
for ticker,features in features_by_target.items():
    for field in feature_sets[ticker]["S2"]:
        s = features[field]
        feature_diagnostics.append(dict(
            ticker=ticker,feature=field,missing_rows=int(s.isna().sum()),
            first_available=s.first_valid_index(),last_available=s.last_valid_index(),
            min=s.min(),median=s.median(),max=s.max()))
feature_diagnostics = pd.DataFrame(feature_diagnostics)
preview_dates = [pd.Timestamp("2008-01-02"),pd.Timestamp("2016-09-19"),
                 pd.Timestamp("2020-03-24"),pd.Timestamp("2026-09-29"),pd.Timestamp("2026-09-30")]
preview_columns = ["date","ticker","own_cumret_20d","own_mom_12_1","own_vol_20d",
                   "own_relative_volume_20d","own_amihud_20d_x1e6","state_VIX_log_level",
                   "cross_XLF_cumret_20d","rel_rank20_9sectors"]
preview = feature_table.loc[feature_table["date"].isin(preview_dates),preview_columns]
metadata_definitions = {
    "date":"New York signal date; part of the primary key",
    "ticker":"Target sector; part of the primary key",
    "split":"Signal-date partition: warmup, train, validation or test",
    "signal_at":"18:00 New York time, assuming daily inputs are available",
    "entry_date":"Next SPY trading date; missing beyond the data cutoff",
    "exit_date":"Second next SPY trading date; missing beyond the data cutoff",
    "entry_at":"Entry open at 09:30 New York time; missing if entry is unavailable",
    "label_available_at":"Exit open at 09:30 New York time; must not exceed the fit cutoff",
    "sector_forward_cash_return":"Sector cash return from the next open to the following open",
    "spy_forward_cash_return":"SPY cash return over the same holding period",
    "target_relative_return":"Sector minus SPY forward cash return, in decimal units",
    "label_valid":"Label is calculable ex post; availability must be checked separately",
    "label_within_split":"Valid label with exit within the signal-date evaluation partition",
    "invalid_reason":"Missing-label or corporate-action reason; not a predictor",
    "common_features_valid":"Intersection of S0/S1/S2 feature validity; training also requires mature labels",
    "common_scoreable":"Intersection of strict evaluation eligibility; not a permanent training filter",
    "reason_code":"Validity reason code; combined with + in the main table; see reason_codes.csv",
    "feature":"Missing predictor from the target-specific S2 list; audit metadata",
    "code":"Individual validity code; not a predictor",
    "scope":"Restriction scope: feature, label, scoring or none",
    "definition":"Meaning of the validity code",
}
metadata_rows = []
for name,table in [("data/interim/prediction_targets.csv", label_table),
                   ("data/interim/feature_validity.csv", feature_validity),
                   ("notebooks/feature_checks/missing_feature_details.csv", missing_feature_details),
                   ("notebooks/feature_checks/reason_codes.csv", reason_dictionary)]:
    for field in table:
        definition = metadata_definitions.get(field)
        if definition is None:
            if field.startswith("features_valid_"):
                definition="All selected predictors are valid, including dates without future labels"
            elif field.startswith("scoreable_"):
                definition="Valid predictors and label within the evaluation partition, excluding warmup"
            else:
                raise KeyError(f"Missing metadata definition: {field}")
        metadata_rows.append(dict(file=name,field=field,
                                  dtype=str(table[field].dtype),definition=definition))
metadata_dictionary = pd.DataFrame(metadata_rows)

feature_report_tables = {
    "feature_catalog.csv":feature_catalog, "metadata_dictionary.csv":metadata_dictionary, "sample_counts.csv":sample_counts,
    "feature_diagnostics.csv":feature_diagnostics,"feature_preview.csv":preview,
    "numeric_spot_checks.csv":numeric_check_table,"causality_checks.csv":causality_checks,
    "label_maturity_checks.csv":boundary_checks,"validation_checks.csv":feature_validation_checks,
    "missing_feature_details.csv":missing_feature_details,"reason_codes.csv":reason_dictionary,
    "common_sample_counts.csv":common_sample_counts,"split_window_daily.csv":split_window_daily,
    "split_window_summary.csv":split_window_summary,"split_unit_checks.csv":split_unit_checks,
}
for name,table in feature_report_tables.items():
    save_csv(table,f"notebooks/feature_checks/{name}", feature_output_records)

reloaded_targets = pd.read_csv(INTERIM/"prediction_targets.csv",low_memory=False,float_precision="round_trip")
assert reloaded_targets.loc[reloaded_targets["entry_date"].isna(),"entry_at"].isna().all()
assert reloaded_targets.loc[reloaded_targets["exit_date"].isna(),"label_available_at"].isna().all()

reloaded_validity = pd.read_csv(INTERIM/"feature_validity.csv", parse_dates=["date"], dtype={"ticker":"str"}, low_memory=False)
pd.testing.assert_frame_equal(feature_validity, reloaded_validity, check_exact=True)


In [ ]:
feature_output_summary = pd.DataFrame(feature_output_records)
feature_output_summary.to_csv(FEATURE_REPORTS/"output_summary.csv", index=False, encoding="utf-8")
feature_manifest = dict(
    version=FEATURE_VERSION, run_utc=datetime.now(timezone.utc).isoformat(),
    python=platform.python_version(), pandas=pd.__version__, numpy=np.__version__,
    source_hashes=source_hashes, audit_source_hashes=audit_source_hashes,
    cleaning_manifest_sha256=cleaning_manifest_hash,
    definition_pdf_sha256=None, definition_note="Definitions are recorded in feature_catalog.csv.",
    splits={"warmup":"2006-2007", "train":"2008-2017", "validation":"2018-2022", "test":"2023-2026-09-30"},
    feature_counts={"S0":57, "S1":97, "S2":102, "physical":107},
    feature_storage_format="csv",
    feature_csv_read_options={"parse_dates":["date"], "dtype":{"ticker":"str"}, "low_memory":False, "float_precision":"round_trip"},
    feature_csv_exact_roundtrip_verified=True,
    validity_schema={"columns":14, "missing_details":"notebooks/feature_checks/missing_feature_details.csv",
                     "reason_dictionary":"notebooks/feature_checks/reason_codes.csv",
                     "common_feature_mask":"common_features_valid", "common_scoring_mask":"common_scoreable"},
    preprocessing="No fitted preprocessing is applied; scaling, clipping and selection use historical training windows only.",
    data_cutoff="2026-09-30", label="sector cash OO(t+1,t+2) - SPY cash OO(t+1,t+2)",
    beta_note="SPY-relative returns retain residual market exposure; beta neutrality is not imposed.",
    validations=int(len(feature_validation_checks)), outputs=feature_output_records,
)
(FEATURE_REPORTS/"feature_manifest.json").write_text(json.dumps(feature_manifest, ensure_ascii=False, indent=2), encoding="utf-8")


## Model Panels

Merge on `(date, ticker)` and export the shared metadata separately from each sector-specific predictor set.

In [ ]:
validation_records = []
panel_features = feature_table
panel_validity = feature_validity
panel_targets = label_table.copy()
for field in TIME_FIELDS:
    panel_targets[field] = pd.to_datetime(panel_targets[field], utc=True).dt.tz_convert(NY)
panel_targets["invalid_reason"] = panel_targets["invalid_reason"].replace("", np.nan)

feature_manifest_path = FEATURE_REPORTS / "feature_manifest.json"
panel_source_names = ["data/interim/sector_features.csv", "data/interim/prediction_targets.csv",
                      "data/interim/feature_validity.csv", "notebooks/feature_checks/feature_sets.json"]
written_hashes = {
    record["file"].replace("\\", "/"): record["sha256"]
    for record in feature_output_records
}
upstream_before = {name:written_hashes[name] for name in panel_source_names}
upstream_before[feature_manifest_path.relative_to(ROOT).as_posix()] = sha256(feature_manifest_path)
upstream_before["data/interim/etf_prices_aligned.csv"] = source_hashes["data/interim/etf_prices_aligned.csv"]

def read_metadata(path):
    frame = pd.read_csv(path, **read_options)
    for field in ["entry_date", "exit_date"]:
        frame[field] = pd.to_datetime(frame[field])
    for field in TIME_FIELDS:
        frame[field] = pd.to_datetime(frame[field], utc=True).dt.tz_convert(NY)
    return frame

In [ ]:
check('source_keys_unique_sorted', all(
    not frame.duplicated(KEYS).any()
    and frame[KEYS].equals(frame.sort_values(KEYS)[KEYS].reset_index(drop=True))
    for frame in [panel_features,panel_targets,panel_validity]), 'Unique ordered date/sector keys')
check('all_source_keys_identical', panel_features[KEYS].equals(panel_targets[KEYS])
      and panel_targets[KEYS].equals(panel_validity[KEYS]), '46,962 matching date/sector keys')
overlap = sorted((set(panel_targets.columns) & set(panel_validity.columns)) - set(KEYS))
check('duplicate_metadata_identical', overlap == ['label_valid','label_within_split','split']
      and panel_targets[overlap].equals(panel_validity[overlap]), overlap)
check('actual_feature_lists_safe', all(
    len(feature_sets[t][s]) == {'S0':57,'S1':97,'S2':102}[s]
    and not any(c.startswith(f'cross_{t}_') for c in feature_sets[t][s])
    and not (set(feature_sets[t][s]) & set(feature_spec['excluded_from_predictors']))
    for t in SECTORS for s in SCHEMES), 'Predictor sets match the feature specification and exclude self cross columns and metadata')

In [ ]:
sample_panel = panel_targets.merge(panel_validity.drop(columns=overlap), on=KEYS,
                             how='outer', validate='one_to_one', indicator=True)
check('metadata_merge_full_coverage', sample_panel['_merge'].eq('both').all()
      and len(sample_panel) == 46962, sample_panel['_merge'].value_counts().to_dict())
sample_panel = sample_panel.drop(columns='_merge').sort_values(KEYS).reset_index(drop=True)
check('shared_panel_schema', sample_panel.shape == (46962,23)
      and not any(c.endswith(('_x','_y')) for c in sample_panel.columns), sample_panel.shape)

### Independent Label Verification

Recompute cash open-to-open labels from entry and exit opens and exit-date dividends.

In [ ]:
label_prices = prices[["date", "ticker", "open", "dividends"]].set_index(KEYS).sort_index()
truth = sample_panel.loc[sample_panel['label_valid']].copy()
def endpoint_values(dates, tickers, field):
    keys = pd.MultiIndex.from_arrays([dates.to_numpy(), np.asarray(tickers)], names=KEYS)
    return label_prices.reindex(keys)[field].to_numpy()

entry = endpoint_values(truth['entry_date'], truth['ticker'], 'open')
exit_open = endpoint_values(truth['exit_date'], truth['ticker'], 'open')
exit_dividend = endpoint_values(truth['exit_date'], truth['ticker'], 'dividends')
spy_tickers = np.repeat('SPY', len(truth))
spy_entry = endpoint_values(truth['entry_date'], spy_tickers, 'open')
spy_exit = endpoint_values(truth['exit_date'], spy_tickers, 'open')
spy_dividend = endpoint_values(truth['exit_date'], spy_tickers, 'dividends')
manual_sector = (exit_open+exit_dividend)/entry - 1
manual_spy = (spy_exit+spy_dividend)/spy_entry - 1
manual_target = manual_sector-manual_spy
truth['independent_abs_error'] = np.abs(manual_target-truth['target_relative_return'].to_numpy())
label_checks = truth.groupby('ticker').agg(
    valid_labels=('target_relative_return','size'),
    max_abs_error=('independent_abs_error','max')).reset_index()
check('independent_open_dividend_labels', len(truth) == 46943
      and np.array_equal(manual_sector,truth['sector_forward_cash_return'].to_numpy())
      and np.array_equal(manual_spy,truth['spy_forward_cash_return'].to_numpy())
      and np.array_equal(manual_target,truth['target_relative_return'].to_numpy()), label_checks.to_dict('records'))
invalid = sample_panel.loc[~sample_panel['label_valid']]
check('invalid_labels_preserved', invalid['invalid_reason'].value_counts().to_dict() == {
    'future_open_outside_data_cutoff':18, 'xlf_in_kind_distribution':1}, invalid['invalid_reason'].value_counts().to_dict())


### Predictor Files

Each file contains the keys and its selected predictors. Exact CSV round trips preserve numeric values and missing observations.

In [ ]:
panel_output_records, roundtrip_rows = [], []

save_csv(sample_panel, 'data/processed/model_panel.csv', panel_output_records, include_bytes=True)
pd.testing.assert_frame_equal(sample_panel, read_metadata(PROCESSED/'model_panel.csv'), check_exact=True)
input_files = {}
for ticker in SECTORS:
    input_files[ticker] = {}
    for scheme in SCHEMES:
        cols = feature_sets[ticker][scheme]
        x = panel_features.loc[panel_features['ticker'].eq(ticker),KEYS+cols].reset_index(drop=True)
        relative_path = f'data/processed/model_inputs/{ticker}_{scheme}.csv'
        save_csv(x, relative_path, panel_output_records, include_bytes=True)
        restored = pd.read_csv(ROOT/relative_path, **read_options)
        pd.testing.assert_frame_equal(restored, x, check_exact=True)
        assert len(x) == len(calendar) == 5218
        input_files[ticker][scheme] = relative_path
        roundtrip_rows.append(dict(ticker=ticker, feature_set=scheme, rows=len(x),
                                   feature_count=len(cols), file_columns=len(x.columns),
                                   missing_cells=int(x[cols].isna().sum().sum()), exact_match=True))
roundtrip_checks = pd.DataFrame(roundtrip_rows)
check('all_27_input_files_exact_and_safe', len(roundtrip_checks) == 27
      and roundtrip_checks['exact_match'].all(), 'Exact feature values and missingness; no targets or self cross predictors')
check('shared_panel_exact_roundtrip', True, 'Exact metadata round trip, including timestamps and missing values')

panel_spec = dict(
    version=PANEL_VERSION, source_feature_version=feature_spec['version'], keys=KEYS,
    model_structure='one_model_per_target_sector',
    shared_panel='data/processed/model_panel.csv',
    shared_panel_role='Shared targets, timestamps and sample validity; contains no predictor features.',
    shared_panel_columns=sample_panel.columns.tolist(),
    input_files=input_files, predictors_by_target=feature_sets,
    predictor_counts={'S0':57,'S1':97,'S2':102},
    target='target_relative_return', signal_time='18:00 America/New_York',
    entry='next SPY trading day open', exit='second next SPY trading day open',
    training_filter=['date >= 2008-01-01','common_features_valid','label_valid',
                     'signal_at <= fit_cutoff','label_available_at <= fit_cutoff'],
    strict_scoring_filter='common_scoreable plus chosen evaluation split',
    sequence_note='Sequence models additionally require a complete L-step input window on the full calendar.',
    read_csv_options={'parse_dates':['date'],'dtype':{'ticker':'str'},'low_memory':False,'float_precision':'round_trip'},
    timestamp_read_rule='pd.to_datetime(field, utc=True).dt.tz_convert(America/New_York)',
    columns_not_predictors=sample_panel.columns.tolist(),
    data_cutoff='2026-09-30', no_fitted_preprocessing=True)
panel_spec_path = PROCESSED/'panel_spec.json'
panel_spec_path.write_text(json.dumps(panel_spec,ensure_ascii=False,indent=2)+'\n',encoding='utf-8')
panel_output_records.append(dict(file=panel_spec_path.relative_to(ROOT).as_posix(), rows=None, columns=None,
                           bytes=panel_spec_path.stat().st_size, sha256=sha256(panel_spec_path)))

## Panel Validation

Check common-sample coverage, holding endpoints and monthly training eligibility against label availability.

In [ ]:
summary_rows = []
for (ticker, split), group in sample_panel.groupby(['ticker','split'], sort=False):
    summary_rows.append(dict(ticker=ticker, split=split, rows=len(group),
        first_signal_date=group['date'].min(), last_signal_date=group['date'].max(),
        labels_valid=int(group['label_valid'].sum()),
        common_features_valid_rows=int(group['common_features_valid'].sum()),
        common_scoreable_rows=int(group['common_scoreable'].sum())))
panel_summary = pd.DataFrame(summary_rows)
totals = panel_summary.groupby('split')[['rows','common_features_valid_rows','common_scoreable_rows']].sum()
check('common_samples_preserved', totals.to_dict('index') == {
    'warmup':{'rows':4518,'common_features_valid_rows':1476,'common_scoreable_rows':0},
    'train':{'rows':22662,'common_features_valid_rows':21930,'common_scoreable_rows':21911},
    'validation':{'rows':11331,'common_features_valid_rows':11331,'common_scoreable_rows':11313},
    'test':{'rows':8451,'common_features_valid_rows':8451,'common_scoreable_rows':8433}}, totals.to_dict('index'))
latest = sample_panel.loc[sample_panel['date'].isin(pd.to_datetime(['2026-09-29','2026-09-30']))]
check('latest_prediction_rows_kept', len(latest) == 18 and latest['common_features_valid'].all()
      and not latest['common_scoreable'].any() and not latest['label_valid'].any(), '18 terminal prediction rows retained without labels')
check('timestamp_clock_and_missing_counts', sample_panel['signal_at'].dt.hour.eq(18).all()
      and sample_panel['entry_at'].isna().sum() == 9 and sample_panel['label_available_at'].isna().sum() == 18
      and sample_panel['entry_at'].dropna().dt.hour.eq(9).all()
      and sample_panel['entry_at'].dropna().dt.minute.eq(30).all()
      and sample_panel['label_available_at'].dropna().dt.hour.eq(9).all()
      and sample_panel['label_available_at'].dropna().dt.minute.eq(30).all(), '18:00 New York signals and 09:30 holding endpoints')
positions = calendar.get_indexer(sample_panel['date'])
entry_expected = pd.Series([calendar[k+1] if k+1 < len(calendar) else pd.NaT for k in positions])
exit_expected = pd.Series([calendar[k+2] if k+2 < len(calendar) else pd.NaT for k in positions])
check('holding_dates_match_calendar',
      np.array_equal(sample_panel['entry_date'].to_numpy(), entry_expected.to_numpy(), equal_nan=True)
      and np.array_equal(sample_panel['exit_date'].to_numpy(), exit_expected.to_numpy(), equal_nan=True), 'Next-day entry and second-next-day exit on the full calendar')

In [ ]:
def training_mask(info, fit_cutoff):
    fit_at = pd.Timestamp(fit_cutoff)
    if fit_at.tzinfo is None:
        raise ValueError('fit_cutoff must be timezone-aware.')
    return (info['date'].ge(pd.Timestamp('2008-01-01'))
            & info['common_features_valid'] & info['label_valid']
            & info['signal_at'].le(fit_at) & info['label_available_at'].le(fit_at))

monthly_rows = []
for month in pd.period_range('2018-01','2026-09',freq='M'):
    month_dates = calendar[(calendar >= month.start_time) & (calendar <= month.end_time)]
    first_signal = month_dates[0]
    fit_date = calendar[calendar.get_loc(first_signal)-1]
    fit_at = (fit_date+pd.Timedelta(hours=18)).tz_localize(NY)
    for ticker in SECTORS:
        info = sample_panel.loc[sample_panel['ticker'].eq(ticker)]
        eligible = info.loc[training_mask(info,fit_at)]
        prediction = info.loc[info['date'].isin(month_dates)]
        monthly_rows.append(dict(prediction_month=str(month), ticker=ticker,
            first_signal_date=first_signal, fit_at=fit_at, training_rows_common=len(eligible),
            latest_training_signal=eligible['date'].max(),
            latest_label_available_at=eligible['label_available_at'].max(),
            eligible_cross_split_rows=int((~eligible['label_within_split']).sum()),
            prediction_rows_common=int(prediction['common_features_valid'].sum()),
            scoreable_rows_common=int(prediction['common_scoreable'].sum())))
monthly_eligibility = pd.DataFrame(monthly_rows)
check('monthly_schedule_size', len(monthly_eligibility) == 105*9, '105 months and nine sectors')
check('monthly_labels_known_at_fit', monthly_eligibility['latest_label_available_at'].le(
      monthly_eligibility['fit_at']).all(), 'All training labels are available at the fitting timestamp')
boundary = monthly_eligibility.loc[monthly_eligibility['prediction_month'].isin(['2018-01','2023-01'])]
check('first_month_label_boundaries',
      boundary.loc[boundary['prediction_month'].eq('2018-01'),'latest_training_signal'].eq(pd.Timestamp('2017-12-27')).all()
      and boundary.loc[boundary['prediction_month'].eq('2023-01'),'latest_training_signal'].eq(pd.Timestamp('2022-12-28')).all(),
      'Month-end fits exclude the two most recent unmatured signal labels')
check('mature_cross_split_labels_not_permanently_dropped',
      monthly_eligibility.loc[monthly_eligibility['prediction_month'].eq('2018-02'),'eligible_cross_split_rows'].eq(2).all(),
      'Mature cross-split labels remain eligible for later training')

In [ ]:
check('all_upstream_files_unchanged', all(sha256(ROOT/p) == h for p,h in upstream_before.items()), len(upstream_before))
check('outputs_have_unique_paths', len(panel_output_records) == len({r['file'] for r in panel_output_records}), len(panel_output_records))
panel_validation_checks = pd.DataFrame(validation_records)
preview_cols = KEYS+['split','target_relative_return','common_features_valid','common_scoreable','reason_code']
panel_preview = sample_panel.loc[sample_panel['date'].isin(pd.to_datetime(
    ['2008-01-02','2016-09-15','2016-09-19','2026-09-29','2026-09-30'])),preview_cols]
for name,frame in {
    'panel_summary.csv':panel_summary, 'csv_roundtrip_checks.csv':roundtrip_checks,
    'independent_label_checks.csv':label_checks, 'monthly_training_eligibility.csv':monthly_eligibility,
    'panel_preview.csv':panel_preview, 'validation_checks.csv':panel_validation_checks,
}.items():
    save_csv(frame, f'notebooks/panel_checks/{name}', panel_output_records, include_bytes=True)

panel_output_summary = pd.DataFrame(panel_output_records)
panel_output_summary.to_csv(PANEL_REPORTS/'output_summary.csv',index=False,encoding='utf-8')
panel_manifest = dict(
    version=PANEL_VERSION, run_utc=datetime.now(timezone.utc).isoformat(),
    python=platform.python_version(),pandas=pd.__version__,numpy=np.__version__,
    source_feature_version=feature_manifest['version'],
    feature_manifest_sha256=sha256(feature_manifest_path),
    protected_upstream_hashes=upstream_before,
    keys=KEYS, rows=46962, rows_per_sector=5218, shared_columns=23, input_files=27,
    predictor_counts={'S0':57,'S1':97,'S2':102},
    export_policy='Shared labels/metadata + per-target per-scheme pure predictors; full panel reconstructed by keys.',
    labels_independently_verified=46943, validations=len(panel_validation_checks),
    no_rows_dropped=True,no_imputation=True,no_fitted_preprocessing=True,
    monthly_schedule='Previous SPY trading day 18:00 New York before each prediction month; expanding training from 2008.',
    outputs=panel_output_records)
(PANEL_REPORTS/'panel_manifest.json').write_text(json.dumps(panel_manifest,ensure_ascii=False,indent=2)+'\n',encoding='utf-8')

## Coverage Summary

In [ ]:
display(panel_summary.groupby('split',sort=False)[
    ['rows','labels_valid','common_features_valid_rows','common_scoreable_rows']].sum())
display(roundtrip_checks.groupby('feature_set')[['rows','feature_count','file_columns']].first())